In [8]:
from pathlib import Path
import json
import shutil
import icechunk as ic
import obstore.store as obs
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser

kerchunk_file = Path("patched_json/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json")
temp_filtered_file = Path("patched_json/filtered_data_only.json")

remote_base = "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6"
ice_prefix = remote_base + "/"

with open(kerchunk_file, "r") as f:
    full_manifest = json.load(f)

refs = full_manifest["refs"]

metadata_refs = {}
chunk_refs = {}
inline_data_refs = {}

for key, val in refs.items():
    # Always keep Zarr metadata
    if key.endswith(".zarray") or key.endswith(".zattrs") or key == ".zgroup":
        metadata_refs[key] = val

    # Keep external chunk byte-range refs
    elif isinstance(val, list) and len(val) == 3 and isinstance(val[0], str):
        path = val[0]

        # Convert GCS refs to HTTPS refs
        path = path.replace(
            "gcs://noaa-oar-cefi-regional-mom6/",
            ice_prefix,
        )

        # If refs were already HTTPS, normalize accidental double slashes
        path = path.replace(remote_base + "//", remote_base + "/")

        # Save patched chunk ref
        chunk_refs[key] = [path, val[1], val[2]]

    # Drop inline/base64/small literal chunks for now
    else:
        inline_data_refs[key] = val

filtered_manifest = {
    **full_manifest,
    "refs": {
        **metadata_refs,
        **chunk_refs,
    },
}

with open(temp_filtered_file, "w") as f:
    json.dump(filtered_manifest, f)

# Local manifest registry
local_dir = temp_filtered_file.parent.resolve()
local_base = local_dir.as_uri() + "/"
absolute_url = temp_filtered_file.resolve().as_uri()

registry = ObjectStoreRegistry()
registry.register(local_base, obs.LocalStore(str(local_dir)))

# Remote chunk registry: no trailing slash here
registry.register(remote_base, obs.HTTPStore(remote_base))

parser = KerchunkJSONParser()

vds = open_virtual_dataset(
    url=absolute_url,
    registry=registry,
    parser=parser,
)

print(vds)

/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


<xarray.Dataset> Size: 972MB
Dimensions:  (lat: 844, lon: 774, time: 372)
Coordinates:
  * lat      (lat) float64 7kB 5.273 5.335 5.398 5.461 ... 58.04 58.1 58.16
  * lon      (lon) float64 6kB -98.0 -97.92 -97.84 ... -36.24 -36.16 -36.08
  * time     (time) datetime64[ns] 3kB NaT NaT NaT NaT NaT ... NaT NaT NaT NaT
Data variables:
    ALB      (time, lat, lon) float32 972MB ManifestArray<shape=(372, 844, 77...
Attributes: (12/25)
    NumFilesInSet:          1
    cefi_archive_version:   /archive/acr/fre/NWA/2025_04/NWA12_COBALT_2025_04...
    cefi_aux:               Postprocessed Data : regrid to regular grid
    cefi_data_doi:          10.5281/zenodo.7893386
    cefi_date_range:        199301-202312
    cefi_ensemble_info:     N/A
    ...                     ...
    cefi_run_xml:           N/A
    cefi_subdomain:         full
    cefi_variable:          ALB
    grid_tile:              N/A
    grid_type:              regular
    title:                  NWA12_COBALT_2025_04_bugfixes


In [ ]:
# Drop broken inline-derived time coords before writing to Icechunk
vds = vds.drop_vars(
    [name for name in ["time", "time_bnds"] if name in vds],
    errors="ignore",
)

In [2]:
config = ic.config.RepositoryConfig.default()

# Icechunk wants trailing slash in url_prefix
container = ic.virtual.VirtualChunkContainer(
    url_prefix=ice_prefix,
    store=ic.storage.http_store(),
)

config.set_virtual_chunk_container(container)

repo_path = Path("my_icechunk_repo")
if repo_path.exists():
    shutil.rmtree(repo_path)

ice_storage = ic.local_filesystem_storage(path=str(repo_path))
repo = ic.Repository.create(ice_storage, config)

session = repo.writable_session(branch="main")
vds.vz.to_icechunk(session.store)
session.commit("Ingested Kerchunk virtual refs into Icechunk")

temp_filtered_file.unlink(missing_ok=True)
print("Icechunk repository created")

  2026-05-28T00:28:08.792968Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196



/srv/conda/envs/notebook/lib/python3.12/site-packages/xarray/coding/times.py:1091: RuntimeWarning: All-NaN slice encountered
  and np.nanmin(dates).astype("=M8[us]").astype(datetime)


TypeError: '<' not supported between instances of 'NoneType' and 'datetime.datetime'

In [4]:
session = repo.writable_session(branch="main")
vds.vz.to_icechunk(session.store)
session.commit("Ingested Kerchunk virtual refs into Icechunk")

'2RDBKSGW2DYKMNQFZY7G'

In [5]:
import icechunk as ic
import zarr

storage = ic.local_filesystem_storage("my_icechunk_repo")

repo = ic.Repository.open(storage)

session = repo.readonly_session(branch="main")

root = zarr.open_group(session.store, mode="r")

print(root.tree())

  2026-05-28T00:31:43.618122Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196



/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


/
├── ALB (372, 844, 774) float32
├── lat (844,) float64
└── lon (774,) float64

In [9]:
from pathlib import Path
import json
import shutil
import icechunk as ic
import obstore.store as obs
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser

kerchunk_file = Path("patched_json/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json")
temp_filtered_file = Path("patched_json/filtered_data_only.json")

remote_base = "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6"
ice_prefix = remote_base + "/"

with open(kerchunk_file, "r") as f:
    full_manifest = json.load(f)

refs = full_manifest["refs"]

KEEP_INLINE_VARS = {"time", "time_bnds", "init", "lead", "member", "month"}

metadata_refs = {}
chunk_refs = {}
inline_data_refs = {}

for key, val in refs.items():
    varname = key.split("/")[0]

    # Always keep Zarr metadata
    if key.endswith(".zarray") or key.endswith(".zattrs") or key == ".zgroup":
        metadata_refs[key] = val

    # Keep inline chunks for selected coordinate/small variables
    elif varname in KEEP_INLINE_VARS:
        chunk_refs[key] = val

    # Keep external chunk byte-range refs
    elif isinstance(val, list) and len(val) == 3 and isinstance(val[0], str):
        path = val[0]

        path = path.replace(
            "gcs://noaa-oar-cefi-regional-mom6/",
            ice_prefix,
        )

        path = path.replace(remote_base + "//", remote_base + "/")

        chunk_refs[key] = [path, val[1], val[2]]

    # Drop all other inline/base64/small literal chunks
    else:
        inline_data_refs[key] = val

# Local manifest registry
local_dir = temp_filtered_file.parent.resolve()
local_base = local_dir.as_uri() + "/"
absolute_url = temp_filtered_file.resolve().as_uri()

registry = ObjectStoreRegistry()
registry.register(local_base, obs.LocalStore(str(local_dir)))

# Remote chunk registry: no trailing slash here
registry.register(remote_base, obs.HTTPStore(remote_base))

parser = KerchunkJSONParser()

vds = open_virtual_dataset(
    url=absolute_url,
    registry=registry,
    parser=parser,
)

print(vds)

<xarray.Dataset> Size: 972MB
Dimensions:  (lat: 844, lon: 774, time: 372)
Coordinates:
  * lat      (lat) float64 7kB 5.273 5.335 5.398 5.461 ... 58.04 58.1 58.16
  * lon      (lon) float64 6kB -98.0 -97.92 -97.84 ... -36.24 -36.16 -36.08
  * time     (time) datetime64[ns] 3kB NaT NaT NaT NaT NaT ... NaT NaT NaT NaT
Data variables:
    ALB      (time, lat, lon) float32 972MB ManifestArray<shape=(372, 844, 77...
Attributes: (12/25)
    NumFilesInSet:          1
    cefi_archive_version:   /archive/acr/fre/NWA/2025_04/NWA12_COBALT_2025_04...
    cefi_aux:               Postprocessed Data : regrid to regular grid
    cefi_data_doi:          10.5281/zenodo.7893386
    cefi_date_range:        199301-202312
    cefi_ensemble_info:     N/A
    ...                     ...
    cefi_run_xml:           N/A
    cefi_subdomain:         full
    cefi_variable:          ALB
    grid_tile:              N/A
    grid_type:              regular
    title:                  NWA12_COBALT_2025_04_bugfixes


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


In [11]:
chunk_refs

{'ALB/0.0.0': ['https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.nc',
  35930,
  683249],
 'ALB/0.0.1': ['https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.nc',
  719179,
  2189765],
 'ALB/0.0.2': ['https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.nc',
  2908944,
  1717497],
 'ALB/0.0.3': ['https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.nc',
  4626441,
  2326770],
 'ALB/0.1.0': ['https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_d

In [12]:
import icechunk as ic
import xarray as xr

storage = ic.s3_storage(
    bucket="icechunk-public-data",
    prefix="v1/era5_weatherbench2",
    region="us-east-1",
    anonymous=True,
)

repo = ic.Repository.open(storage=storage)
session = repo.readonly_session("main")
ds = xr.open_dataset(
    session.store, group="1x721x1440", engine="zarr", chunks=None, consolidated=False
)

In [13]:
ds

<xarray.Dataset> Size: 7TB
Dimensions:                  (time: 561264, latitude: 721, longitude: 1440)
Coordinates:
  * time                     (time) datetime64[ns] 4MB 1959-01-01 ... 2023-01...
  * latitude                 (latitude) float32 3kB 90.0 89.75 ... -89.75 -90.0
  * longitude                (longitude) float32 6kB 0.0 0.25 ... 359.5 359.8
Data variables:
    10m_u_component_of_wind  (time, latitude, longitude) float32 2TB ...
    10m_v_component_of_wind  (time, latitude, longitude) float32 2TB ...
    2m_temperature           (time, latitude, longitude) float32 2TB ...
Attributes:
    selector:  {'time': slice(None, None, None)}

In [14]:
import icechunk as ic
import zarr

storage = ic.r2_storage(
    prefix="v1/era5_weatherbench2",
    endpoint_url="https://data.icechunk.cloud",
    anonymous=True,
)

repo = ic.Repository.open(storage=storage)
session = repo.readonly_session("main")

root = zarr.open_group(session.store, mode="r")
print(root.tree())

/
└── 1x721x1440
    ├── 10m_u_component_of_wind (561264, 721, 1440) float32
    ├── 10m_v_component_of_wind (561264, 721, 1440) float32
    ├── 2m_temperature (561264, 721, 1440) float32
    ├── latitude (721,) float32
    ├── longitude (1440,) float32
    └── time (561264,) int64